# BioQEC: Stim, PyMatching e ruído não estacionário

Este notebook implementa um teste reprodutível das **Etapas 1 e 2** do BioQEC:

1. geração de circuitos de memória de códigos de superfície rotacionados;
2. amostragem de síndromes e observáveis lógicos;
3. decodificação MWPM;
4. geração de cenários estacionários, abruptos, com deriva, recorrência e OOD;
5. comparação pareada entre um decoder estático e um decoder-oráculo;
6. monitoramento exploratório por CUSUM.

> **Limitação:** o gerador padrão do Stim usa parâmetros de ruído estáticos por circuito. Assim, a trajetória não estacionária é aproximada por circuitos independentes em janelas. Isso testa o descasamento de pesos e o pipeline experimental, mas não representa ainda uma única memória lógica contínua com $p(t)$ variando a cada ciclo.

## 1. Instalação das dependências

In [ ]:
# Execute esta célula no Colab, Jupyter ou GitHub Codespaces.
%pip install -q stim==1.16.0 pymatching==2.4.0 numpy pandas scipy matplotlib pytest

## 2. Importação do projeto

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stim
import pymatching

from circuits.surface_code import (
    CircuitConfig,
    build_memory_circuit,
    circuit_metadata,
    sample_syndromes,
)
from decoders.mwpm import build_matching, decode_batch
from experiments.windowed_protocol import run_windowed_benchmark
from monitoring.change_detection import CUSUMConfig, fit_reference, upper_cusum

from features.causal_features import extract_causal_features
from monitoring.novelty import ChangeNoveltyCoordinator, NoveltyConfig
from noise.ood import (TrainingSupport, TrajectoryDescriptor, assess_trajectory, classify_regime)
from decoders.adaptive_mwpm import (
    PeriodicRecalibrationConfig,
    should_recalibrate,
    shrink_probabilities,
    probabilities_to_weights,
)
from selection.pareto import CandidateMetrics, OperationalLimits, pareto_front

from noise.nonstationary import (
    default_regimes,
    make_abrupt_change,
    make_drift,
    make_ood,
    make_recurrence,
)

print("Stim:", stim.__version__)
print("PyMatching:", pymatching.__version__)

## 3. Circuito estacionário de referência

In [ ]:
cfg = CircuitConfig(
    distance=5,
    rounds=5,
    basis="X",
    noise_model="depolarizing",
    p=1e-3,
    seed=42,
)

circuit = build_memory_circuit(cfg)
print(cfg.label)
print(circuit_metadata(circuit))
print("\nPrimeiras instruções do circuito:\n")
print("\n".join(str(circuit).splitlines()[:25]))

## 4. Amostragem e decodificação MWPM

In [ ]:
detections, observables = sample_syndromes(
    circuit,
    n_shots=5_000,
    seed=cfg.seed,
)
matching = build_matching(circuit)
result = decode_batch(matching, detections, observables)

print("Shape das síndromes:", detections.shape)
print("Shape dos observáveis:", observables.shape)
print(f"LER estimada: {result.logical_error_rate:.6f}")
print(f"Latência do decode_batch: {result.microseconds_per_shot:.3f} us/shot")

## 5. Validação básica por distância e intensidade

In [ ]:
rows = []
for distance in [3, 5, 7]:
    for p in [1e-3, 3e-3, 7e-3]:
        local_cfg = CircuitConfig(
            distance=distance,
            rounds=distance,
            basis="X",
            p=p,
            seed=1000 + distance,
        )
        local_circuit = build_memory_circuit(local_cfg)
        det, obs = sample_syndromes(local_circuit, n_shots=3_000, seed=local_cfg.seed)
        local_result = decode_batch(build_matching(local_circuit), det, obs)
        rows.append({
            "distance": distance,
            "p": p,
            "shots": det.shape[0],
            "logical_failures": int(local_result.logical_failures.sum()),
            "LER": local_result.logical_error_rate,
            "us_per_shot": local_result.microseconds_per_shot,
        })

static_df = pd.DataFrame(rows)
static_df

In [ ]:
plot_df = static_df.copy()
plot_df["LER_plot"] = plot_df["LER"].clip(lower=0.5 / plot_df["shots"])
for p, group in plot_df.groupby("p"):
    plt.plot(group["distance"], group["LER_plot"], marker="o", label=f"p={p:g}")
plt.yscale("log")
plt.xlabel("Distância do código")
plt.ylabel("Taxa de erro lógico (limite visual para zero falhas)")
plt.title("Teste exploratório de escalonamento")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Trajetórias não estacionárias

In [ ]:
regimes = default_regimes()
T = 60
trajectories = {
    "abrupta": make_abrupt_change(regimes["low"], regimes["high"], T=T, seed=10),
    "deriva": make_drift(regimes["low"], regimes["high"], T=T, seed=11),
    "recorrência": make_recurrence(regimes["low"], regimes["high"], T=T, seed=12),
    "OOD": make_ood(regimes["low"], T=T, ood_multiplier=4.0, seed=13),
}

for name, trajectory in trajectories.items():
    plt.plot(np.arange(T), trajectory.p_sequence, label=name)
plt.xlabel("Ciclo")
plt.ylabel("Probabilidade nominal p(t)")
plt.title("Trajetórias latentes de ruído")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Benchmark em janelas: decoder estático versus oráculo

O decoder estático permanece calibrado no regime `low`. O decoder-oráculo recebe o modelo nominal da janela. Ambos decodificam **as mesmas síndromes**, preservando o pareamento.

In [ ]:
trajectory = trajectories["OOD"]
window_results = run_windowed_benchmark(
    trajectory,
    calibration_regime=regimes["low"],
    distance=5,
    basis="X",
    window_size=5,
    shots_per_window=3_000,
)
window_df = pd.DataFrame([row.to_dict() for row in window_results])
window_df

In [ ]:
plt.plot(window_df["start"], window_df["static_ler"], marker="o", label="MWPM estático")
plt.plot(window_df["start"], window_df["oracle_ler"], marker="s", label="MWPM-oráculo")
for cp in trajectory.change_points:
    plt.axvline(cp, linestyle="--", alpha=0.6)
plt.xlabel("Início da janela")
plt.ylabel("Taxa de erro lógico")
plt.title("Custo do descasamento do modelo")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Monitor CUSUM sobre a densidade de detecções

In [ ]:
baseline_windows = max(3, len(window_df) // 3)
reference_mean, reference_std = fit_reference(
    window_df.loc[: baseline_windows - 1, "detection_density"].to_numpy()
)

cusum = upper_cusum(
    window_df["detection_density"].to_numpy(),
    CUSUMConfig(
        reference_mean=reference_mean,
        reference_std=reference_std,
        allowance=0.5,
        threshold=5.0,
    ),
)
window_df["cusum_score"] = cusum.scores
window_df["alarm"] = cusum.alarms
window_df[["window", "start", "detection_density", "cusum_score", "alarm"]]

In [ ]:
plt.plot(window_df["start"], window_df["cusum_score"], marker="o")
for cp in trajectory.change_points:
    plt.axvline(cp, linestyle="--", alpha=0.6)
for start in window_df.loc[window_df["alarm"], "start"]:
    plt.axvline(start, linestyle=":", alpha=0.6)
plt.xlabel("Início da janela")
plt.ylabel("Escore CUSUM")
plt.title("Detecção causal exploratória")
plt.grid(True, alpha=0.3)
plt.show()

## 9. Vetor causal de oito características

A célula usa um fluxo sintético com eixo `(tempo, verificadores)` para tornar explícita a causalidade. O resultado em `t` deve permanecer inalterado quando apenas o futuro é modificado.

In [ ]:
rng = np.random.default_rng(123)
stream = rng.binomial(1, 0.08, size=(30, 6))
check_types = ["X", "X", "X", "Z", "Z", "Z"]
adjacency = [(0, 1), (1, 2), (3, 4), (4, 5), (1, 4)]
analog = np.clip(0.5 + rng.normal(0, 0.25, size=stream.shape), 0, 1)
leakage = rng.binomial(1, 0.01, size=stream.shape)
predictive = np.tile(np.array([0.92, 0.04, 0.03, 0.01]), (stream.shape[0], 1))

f_t = extract_causal_features(
    stream, check_types, adjacency, t=19, window=12,
    leakage=leakage, analog_prob_one=analog, predictive_probs=predictive,
)

stream_changed = stream.copy()
stream_changed[20:] = 1 - stream_changed[20:]
f_t_future_changed = extract_causal_features(
    stream_changed, check_types, adjacency, t=19, window=12,
    leakage=leakage, analog_prob_one=analog, predictive_probs=predictive,
)

assert np.allclose(f_t.values, f_t_future_changed.values, equal_nan=True)
pd.Series(f_t.as_dict(), name="valor em t=19")

## 10. Coordenação entre detecção de mudança e novidade

O CUSUM abre o episódio. A distância à memória apenas classifica o episódio após um período de graça; novos cruzamentos não criam alarmes concorrentes.

In [ ]:
coordinator = ChangeNoveltyCoordinator(
    NoveltyConfig(
        cusum_threshold=5.0, reset_threshold=1.0, grace_cycles=2,
        novelty_threshold=9.0, novelty_window=3, novelty_fraction=2/3,
        reset_cycles=3,
    )
)

example_scores = [0, 2, 6, 7, 8, 7, 4, 0.5, 0.4, 0.3]
example_distances = [1, 1, 12, 13, 11, 12, 3, 2, 1, 1]
state_rows = []
for cycle, (score, distance) in enumerate(zip(example_scores, example_distances)):
    step = coordinator.step(score, distance)
    state_rows.append({
        "cycle": cycle, "cusum": score, "distance": distance,
        "state": step.state.value, "event_id": step.event_id,
        "change_started": step.change_started,
        "classified_now": step.classified_now,
        "novelty_score": step.novelty_score,
    })
pd.DataFrame(state_rows)

## 11. Rótulo OOD independente do BioQEC

O rótulo é construído a partir do suporte declarado de treinamento. O limiar de novidade do método não participa dessa definição.

In [ ]:
support = TrainingSupport(
    noise_models=frozenset({"depolarizing", "measurement_heavy"}),
    p_min=5e-4, p_max=5e-3, max_bias_eta=1.0, max_spatial_delta=0.0,
    seen_compositions=frozenset({frozenset({"depolarizing", "measurement_heavy"})}),
    max_drift_rate=1e-4, max_burst_duration=8, max_persistence=20,
)

ood_rows = []
for name, regime in regimes.items():
    ood_rows.append({
        "regime": name, "p": regime.p, "modelo": regime.noise_model,
        "categoria_isolada": classify_regime(regime, support).value,
    })

held_out = TrajectoryDescriptor(
    regimes=(regimes["low"], regimes["meas"]),
    active_mechanisms=frozenset({"depolarizing", "measurement_heavy"}),
    drift_rate=2e-4,
    burst_duration=12,
)
print("Causas OOD da trajetória:", [c.value for c in assess_trajectory(held_out, support)])
pd.DataFrame(ood_rows)

## 12. Baseline MWPM com recalibração periódica

A agenda e a contração são fixadas antes do teste. A célula ilustra a transformação de probabilidades estimadas em pesos de matching.

In [ ]:
periodic_cfg = PeriodicRecalibrationConfig(
    period=8, calibration_window=16, shrinkage_to_initial=0.2
)
initial_p = np.array([1e-3, 2e-3, 1.5e-3])
estimated_p = np.array([4e-3, 7e-3, 5e-3])
shrunk_p = shrink_probabilities(estimated_p, initial_p, periodic_cfg)
weights = probabilities_to_weights(shrunk_p)

pd.DataFrame({
    "edge": np.arange(len(initial_p)),
    "initial_p": initial_p,
    "estimated_p": estimated_p,
    "shrunk_p": shrunk_p,
    "weight": weights,
}), [cycle for cycle in range(1, 25) if should_recalibrate(cycle, periodic_cfg)]

## 13. Promoção multiobjetivo por Pareto

Latência e custo são restrições, não parcelas somadas ao erro lógico. Entre os candidatos viáveis, preserva-se o conjunto não dominado.

In [ ]:
limits = OperationalLimits(
    latency_p99=100.0, cost=1.0, escalation_rate=0.20, instability=0.10
)
candidates = [
    CandidateMetrics("ativo", 2.0e-3, 0.08, 0.04, 12, 65, 0.6, 0.08, 0.03),
    CandidateMetrics("rápido", 2.2e-3, 0.07, 0.04, 10, 45, 0.5, 0.06, 0.04),
    CandidateMetrics("preciso", 1.6e-3, 0.06, 0.03, 11, 90, 0.8, 0.12, 0.05),
    CandidateMetrics("inviável", 1.2e-3, 0.05, 0.02, 8, 140, 1.2, 0.25, 0.15),
]
front = pareto_front(candidates, limits, tolerances=(1e-5, 1e-3, 1e-3, 0.1))
[name.name for name in front]

## 14. Exportação dos resultados

In [ ]:
output_dir = ROOT / "results"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "notebook_ood_change.csv"
window_df.to_csv(output_path, index=False)
print(output_path.resolve())

## 15. Testes automatizados

In [ ]:
# Execute para validar o repositório.
!pytest -q

## Limites e próxima extensão

A trajetória em janelas não é uma memória lógica contínua com canais variáveis a cada ciclo. A extensão confirmatória deve construir um circuito Stim customizado ou consumir um fluxo temporal de hardware, preservando detectores, observáveis, causalidade e pareamento.